In [1]:
%pwd

'd:\\Revision\\Covid-19_detection_with_MLOPS\\research'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'd:\\Revision\\Covid-19_detection_with_MLOPS'

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    unzip_data_dir: Path
    STATUS_FILE: str
    all_classes: list

In [7]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=Path(config.root_dir),
            unzip_data_dir=Path(config.unzip_data_dir),
            STATUS_FILE=config.STATUS_FILE,
            all_classes=config.all_classes
        )

        return data_validation_config

     

In [8]:
import os
from cnnClassifier import logger

class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_classes_exist(self) -> bool:
        """
        Validates if all expected class directories exist in the ingested dataset.
        Writes 'Validation status: True/False' to the status file.
        """
        try:
            validation_status = True
            
            # List all directories in the ingested dataset path
            available_classes = os.listdir(self.config.unzip_data_dir)

            for expected_cls in self.config.all_classes:
                if expected_cls not in available_classes:
                    validation_status = False
                    logger.error(f"Missing expected class directory: {expected_cls}")
                    break
            
            # Write the final status to the status file
            with open(self.config.STATUS_FILE, 'w') as f:
                f.write(f"Validation status: {validation_status}")
            
            if validation_status:
                logger.info("All expected classes found. Data validation successful.")
            else:
                logger.warning("Data validation failed. Expected classes are missing.")

            return validation_status
            
        except Exception as e:
            logger.exception("Error occurred during data validation.")
            raise e

In [9]:
STAGE_NAME = "Data Validation Stage"
try:
    logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_all_classes_exist()
    logger.info(f">>>>>> stage {STAGE_NAME} completed <<<<<<\n\nx==========x")
except Exception as e:
    logger.exception(e)
    raise e

[2026-06-27 11:22:37,538: INFO: 2320197787: >>>>>> stage Data Validation Stage started <<<<<<]
[2026-06-27 11:22:37,557: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-27 11:22:37,560: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-27 11:22:37,562: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-06-27 11:22:37,564: INFO: common: created directory at: artifacts]
[2026-06-27 11:22:37,564: INFO: common: created directory at: artifacts/data_validation]
[2026-06-27 11:22:37,566: INFO: 585940061: All expected classes found. Data validation successful.]
[2026-06-27 11:22:37,566: INFO: 2320197787: >>>>>> stage Data Validation Stage completed <<<<<<

x==========x]
